In [5]:
import os
import json
import pandas as pd
import concurrent.futures
import threading
import sys
from tqdm import tqdm
import sys
sys.path.append("../")
from src.llama_sft_function_calls import *
from src.load_data import load_data_from_csv

# --- CONFIGURATION ---
BASE_DIR = "../"
DATA_PATH = os.path.join(BASE_DIR, "data", "test_sent_emo.csv")
BIO_CARDS_PATH = os.path.join(BASE_DIR, "logs", "speaker_bio_cards.json")
OUTPUT_FILE = os.path.join(BASE_DIR, "logs", "llama3", "council_llama3_1_results.csv")

In [6]:
# --- UTILS ---
def load_speaker_bio_cards(json_file=BIO_CARDS_PATH):
    try:
        with open(json_file, 'r', encoding='utf-8') as f:
            return json.load(f)
    except Exception as e:
        print(f"Bio cards error: {e}")
        return {}

def get_speaker_bio_card_text(speaker_name, bio_cards_dict):
    if not bio_cards_dict or speaker_name not in bio_cards_dict:
        return None
    bio_card = bio_cards_dict[speaker_name]
    if isinstance(bio_card, dict):
        return "\n".join([f"{k}: {v}" for k, v in bio_card.items()])
    return str(bio_card)

def extract_json_field(raw_str, field):
    """
    Extract JSON field from raw model output with robust parsing.
    Handles markdown code fences and incomplete JSON.
    """
    import re
    try:
        raw_str = str(raw_str).strip()
        
        # Strip markdown code fences if present
        raw_str = re.sub(r'^```json\s*', '', raw_str, flags=re.MULTILINE)
        raw_str = re.sub(r'\s*```\s*$', '', raw_str, flags=re.MULTILINE)
        raw_str = raw_str.strip()
        
        # Find JSON using brace matching (more reliable than regex)
        start_idx = raw_str.find('{')
        if start_idx == -1:
            return "unknown"
        
        brace_count = 0
        end_idx = -1
        
        for i in range(start_idx, len(raw_str)):
            if raw_str[i] == '{':
                brace_count += 1
            elif raw_str[i] == '}':
                brace_count -= 1
                if brace_count == 0:
                    end_idx = i + 1
                    break
        
        if end_idx == -1:
            # Incomplete JSON - try to fix by adding closing braces
            brace_count = raw_str[start_idx:].count('{') - raw_str[start_idx:].count('}')
            json_str = raw_str[start_idx:] + '}' * brace_count
        else:
            json_str = raw_str[start_idx:end_idx]
        
        # Parse JSON
        data = json.loads(json_str)
        value = data.get(field, "unknown")
        
        # Type conversion for numeric fields
        if field == "confidence" and isinstance(value, (int, float)):
            return float(value)
        return value
        
    except json.JSONDecodeError as e:
        print(f"JSON decode error for field '{field}': {e} | First 200 chars: {raw_str[:200]}", flush=True)
        return "unknown"
    except Exception as e:
        print(f"JSON extraction error for field '{field}': {e}", flush=True)
        return "unknown"

In [ ]:
# --- PIPELINE ---
def run_llama3_council_scene(scene_obj, bio_cards, global_history, csv_lock, output_file):
    """
    Process a scene using Phase 3 approach: parallelize across utterances.
    Saves entire scene to CSV after all utterances are processed.
    """
    utterances = scene_obj["utterances"]
    dialogue_id = scene_obj["dialogue_id"]
    scene_script = "\n".join([f"{u['Speaker']}: {u['Utterance']}" for u in utterances])

    # Level 1: Global Context
    global_context = call_llama3_context_manager(scene_script)
    social_graph = call_llama3_relational_graph(scene_script)

    # Define worker function for each utterance
    def process_utterance_with_retry(idx, u, max_retries=3):
        """Process a single utterance with retry logic for 500 errors."""
        rec_id = u.get('Recognition_ID', "unknown_id")
        for attempt in range(max_retries):
            try:
                target_text = u.get('Utterance', "")
                speaker = u.get('Speaker', "Unknown")
                actual_emotion = u.get('Emotion', 'unknown')
                
                previous_turn = utterances[idx - 1] if idx > 0 else None
                prev_text = previous_turn.get('Utterance', "") if previous_turn else ""
                prev_speaker = previous_turn.get('Speaker', "Unknown") if previous_turn else "Unknown"

                # Level 2: Specialists
                profile = call_llama3_profiler(target_text, social_graph)
                sentiment = call_llama3_sentiment(target_text)
                shift = call_llama3_emotional_shift(prev_text, prev_speaker, target_text, speaker, global_context)
                dynamics = call_llama3_social_dynamics(target_text, profile, social_graph)

                # Level 3: Aggregator
                bio_card = get_speaker_bio_card_text(speaker, bio_cards)
                prev_preds = global_history[-3:] if global_history else None
                
                raw_final = call_llama3_council_aggregator(
                    rec_id, target_text, global_context, profile, sentiment, dynamics, shift,
                    speaker_bio_card=bio_card, previous_predictions=prev_preds
                )
                
                return {
                    "Dialogue_ID": dialogue_id,
                    "Recognition_ID": rec_id,
                    "Speaker": speaker,
                    "Utterance": target_text,
                    "Predicted_Emotion_Raw": raw_final,
                    "Actual_Emotion": actual_emotion,
                    "predicted_emotion": extract_json_field(raw_final, "predicted_emotion"),
                    "confidence": extract_json_field(raw_final, "confidence"),
                    "emotional_shift_report": shift
                }
                
            except Exception as e:
                error_msg = str(e).lower()
                if "500" in error_msg or "internal server" in error_msg:
                    if attempt < max_retries - 1:
                        wait_time = 2 ** attempt
                        import time
                        time.sleep(wait_time)
                        continue
                raise Exception(f"{rec_id}: {str(e)}")
        
        raise Exception(f"{rec_id}: Failed after {max_retries} retries")

    # -- PROCESS ALL UTTERANCES IN PARALLEL --
    scene_results = []
    max_workers_utterance = min(5, len(utterances))
    
    with concurrent.futures.ThreadPoolExecutor(max_workers=max_workers_utterance) as executor:
        futures = {
            executor.submit(process_utterance_with_retry, idx, u): idx 
            for idx, u in enumerate(utterances)
        }
        
        for future in concurrent.futures.as_completed(futures):
            try:
                result = future.result()
                scene_results.append(result)
                
                # Update global history
                shift_flag = "TRUE" if "SHIFT: TRUE" in result.get("emotional_shift_report", "") else "FALSE"
                global_history.append({
                    "utterance": result["Utterance"],
                    "emotion": result.get("predicted_emotion", "unknown"),
                    "shift": shift_flag
                })
                if len(global_history) > 3:
                    global_history.pop(0)
                    
            except Exception as e:
                print(f"  ✗ {str(e)[:80]}", flush=True)
                continue
    
    # Save entire scene to CSV (thread-safe)
    if scene_results:
        with csv_lock:
            results_df = pd.DataFrame(scene_results)
            file_exists = os.path.exists(output_file)
            results_df.to_csv(output_file, mode='a', index=False, header=not file_exists)
    
    return scene_results

In [ ]:
print("Starting Llama 3.1 MELD Council Pipeline (with retry logic & scene-level logging)...", flush=True)
print(f"Output: {OUTPUT_FILE}\n", flush=True)

# 1. Load Data
df = load_data_from_csv(DATA_PATH)
df['Recognition_ID'] = df['Dialogue_ID'].astype(str) + "_" + df['Utterance_ID'].astype(str)

# Preprocess into scenes
scenes = []
for diag_id, group in df.groupby('Dialogue_ID'):
    scenes.append({
        "dialogue_id": diag_id,
        "utterances": group[['Utterance', 'Speaker', 'Recognition_ID', 'Emotion']].to_dict(orient='records')
    })

# 2. Check for existing results (Resume capability)
processed_dialogue_ids = set()
if os.path.exists(OUTPUT_FILE):
    try:
        existing_df = pd.read_csv(OUTPUT_FILE)
        if 'Dialogue_ID' in existing_df.columns:
            processed_dialogue_ids = set(existing_df['Dialogue_ID'].unique())
            print(f"Resuming: {len(processed_dialogue_ids)} scenes already processed\n", flush=True)
    except Exception as e:
        print(f"Starting fresh\n", flush=True)

# 3. Process scenes
bio_cards = load_speaker_bio_cards()
global_history = []
csv_lock = threading.Lock()

unprocessed_scenes = [s for s in scenes if s['dialogue_id'] not in processed_dialogue_ids]
print(f"Processing {len(unprocessed_scenes)} scenes...\n", flush=True)

for scene in tqdm(unprocessed_scenes, desc="Scenes", ncols=80):
    try:
        results = run_llama3_council_scene(scene, bio_cards, global_history, csv_lock, OUTPUT_FILE)
        print(f"✓ Scene {scene['dialogue_id']}: {len(results)} utterances", flush=True)
    except Exception as e:
        print(f"✗ Scene {scene['dialogue_id']}: {str(e)[:80]}", flush=True)
        continue

print(f"\n✅ Pipeline finished. Results saved to {OUTPUT_FILE}", flush=True)

Starting Llama 3.1 MELD Council Pipeline - VERSION 1.0.1 (Actual_Emotion fix)...
Data Path: ../data\test_sent_emo.csv
Output File: ../logs\llama3\council_llama3_1_results.csv
Data loaded successfully from ../data\test_sent_emo.csv
Processing 280 scenes.


Processing Scenes:   0%|          | 0/280 [00:00<?, ?it/s]

Calling Context Manager for Scene 0...
Calling Relational Graph for Scene 0...
Processing Utterance 1/3: 0_0
Processing Utterance 2/3: 0_1
Processing Utterance 3/3: 0_2
Aggregator result for 0_2: ```json
{
  "recognition_id": "0_2",
  "reasoning": "All five expert agents concur that the utteranc...
Aggregator result for 0_1: ```json
{
  "recognition_id": "0_1",
  "reasoning": "Rachel's delivery is consistently described as ...
Error in result processing: Error processing utterance 0 (rec_id=0_0): 500 Internal error encountered.
Saved Scene 0 to ../logs\llama3\council_llama3_1_results.csv


Processing Scenes:   0%|          | 1/280 [08:18<38:38:19, 498.56s/it]

Calling Context Manager for Scene 1...
Calling Relational Graph for Scene 1...
Processing Utterance 1/8: 1_0
Processing Utterance 2/8: 1_1
Processing Utterance 3/8: 1_2
Processing Utterance 4/8: 1_3
Processing Utterance 5/8: 1_4
Aggregator result for 1_3: ```json
{
  "recognition_id": "1_3",
  "reasoning": "The utterance 'Push 'em out, push 'em out, way ...
Processing Utterance 6/8: 1_5
Aggregator result for 1_0: ```json
{
  "recognition_id": "1_0",
  "reasoning": "The utterance expresses strong positive sentime...
Processing Utterance 7/8: 1_6
Aggregator result for 1_2: ```json
{
  "recognition_id": "1_2",
  "reasoning": "The utterance expresses strong positive valence...
Processing Utterance 8/8: 1_7
Aggregator result for 1_5: ```json
{
  "recognition_id": "1_5",
  "reasoning": "The utterance 'Let's— I was just—yeah, right.' ...
Error in result processing: Error processing utterance 4 (rec_id=1_4): 500 Internal error encountered.
Error in result processing: Error processing utterance

Processing Scenes:   1%|          | 2/280 [25:29<62:39:40, 811.44s/it]

Calling Context Manager for Scene 2...
Calling Relational Graph for Scene 2...
Processing Utterance 1/11: 2_0
Processing Utterance 2/11: 2_1
Processing Utterance 3/11: 2_2
Processing Utterance 4/11: 2_3
Processing Utterance 5/11: 2_4
Aggregator result for 2_0: {
  "recognition_id": "2_0",
  "reasoning": "The utterance 'Okay.' is consistently categorized as ne...
Processing Utterance 6/11: 2_5
Aggregator result for 2_5: ```json
{
  "recognition_id": "2_5",
  "reasoning": "The Context Historian and Sentiment Analyst con...
Processing Utterance 7/11: 2_6
